In [0]:
# Set the catalog
spark.sql("USE CATALOG final_assesment")

# Set the schema
spark.sql("USE SCHEMA gold")

# Verify the current catalog and schema
print("Current Catalog:", spark.sql("SELECT current_catalog()").first()[0])
print("Current Schema:", spark.sql("SELECT current_schema()").first()[0])

In [0]:
# Read the Silver table
df_silver = spark.table("final_assesment.silver.orders_clean")

# Display schema
df_silver.printSchema()

# Display sample records
display(df_silver.limit(10))

# Check total record count
print("Total Silver Records:", df_silver.count())

### Creation of Customer Dimension

In [0]:
# Select unique customer records
df_customer = (
    df_silver
    .select(
        "customer_id",
        "customer_name",
        "customer_city",
        "customer_state",
        "customer_dob"
    )
    .dropDuplicates(["customer_id"])
)

# Check customer count
print("Total Unique Customers:", df_customer.count())

display(df_customer)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS final_assesment.gold.dim_customer (
    customer_key BIGINT GENERATED ALWAYS AS IDENTITY,
    customer_id STRING NOT NULL,
    customer_name STRING,
    customer_city STRING,
    customer_state STRING,
    customer_dob DATE,

    CONSTRAINT pk_dim_customer
        PRIMARY KEY (customer_key)
)
USING DELTA;

In [0]:
df_customer.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("final_assesment.gold.dim_customer")

In [0]:
%sql
SELECT *
FROM final_assesment.gold.dim_customer
ORDER BY customer_key;

### Creation of Product Dimension

In [0]:
df_product = (
    df_silver
    .select(
        "product_id",
        "product_name",
        "category",
        "brand"
    )
    .dropDuplicates(["product_id"])
)

print("Total Unique Products:", df_product.count())

display(df_product)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS final_assesment.gold.dim_product (
    product_key BIGINT GENERATED ALWAYS AS IDENTITY,
    product_id STRING NOT NULL,
    product_name STRING,
    category STRING,
    brand STRING,

    CONSTRAINT pk_dim_product
        PRIMARY KEY (product_key)
)
USING DELTA;

In [0]:
df_product.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("final_assesment.gold.dim_product")

In [0]:
%sql
SELECT *
FROM final_assesment.gold.dim_product
ORDER BY product_key;

### Creation of Payment Dimension

In [0]:
df_payment = (
    df_silver
    .select(
        "payment_method_id",
        "payment_method",
        "payment_provider"
    )
    .dropDuplicates(["payment_method_id"])
)

print("Total Unique Payment Methods:", df_payment.count())

display(df_payment)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS final_assesment.gold.dim_payment (
    payment_key BIGINT GENERATED ALWAYS AS IDENTITY,
    payment_method_id STRING NOT NULL,
    payment_method STRING,
    payment_provider STRING,

    CONSTRAINT pk_dim_payment
        PRIMARY KEY (payment_key)
)
USING DELTA;

In [0]:
df_payment.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("final_assesment.gold.dim_payment")

In [0]:
%sql
SELECT *
FROM final_assesment.gold.dim_payment
ORDER BY payment_key;

### Create of Date Dimension

In [0]:
from pyspark.sql.functions import (
    col,
    min,
    max,
    sequence,
    explode,
    date_format,
    year,
    quarter,
    month,
    dayofmonth,
    dayofweek,
    dayofyear,
    weekofyear,
    date_format,
    when,
    lit
)

# Find minimum and maximum order dates
date_range = df_silver.agg(
    min("order_date").alias("min_date"),
    max("order_date").alias("max_date")
).first()

min_date = date_range["min_date"]
max_date = date_range["max_date"]

print("Minimum Order Date:", min_date)
print("Maximum Order Date:", max_date)

# Generate all dates in the range
df_dates = spark.sql(f"""
    SELECT explode(
        sequence(
            DATE('{min_date}'),
            DATE('{max_date}'),
            INTERVAL 1 DAY
        )
    ) AS full_date
""")

display(df_dates)

In [0]:
df_dim_date = (
    df_dates
    .withColumn("date_key", date_format(col("full_date"), "yyyyMMdd").cast("int"))
    .withColumn("year", year(col("full_date")))
    .withColumn("quarter", quarter(col("full_date")))
    .withColumn("month", month(col("full_date")))
    .withColumn("month_name", date_format(col("full_date"), "MMMM"))
    .withColumn("day", dayofmonth(col("full_date")))
    .withColumn("day_of_week", dayofweek(col("full_date")))
    .withColumn("day_name", date_format(col("full_date"), "EEEE"))
    .withColumn("day_of_year", dayofyear(col("full_date")))
    .withColumn("week_of_year", weekofyear(col("full_date")))
    .withColumn(
        "is_weekend",
        when(dayofweek(col("full_date")).isin(1, 7), True).otherwise(False)
    )
    .select(
        "date_key",
        "full_date",
        "year",
        "quarter",
        "month",
        "month_name",
        "day",
        "day_of_week",
        "day_name",
        "day_of_year",
        "week_of_year",
        "is_weekend"
    )
)

display(df_dim_date)